In [ ]:
import json







import os







import re







from pathlib import Path















import requests







from dotenv import load_dotenv







from neo4j import GraphDatabase















load_dotenv('.env')















NEO4J_URI = os.getenv('NEO4J_URI', 'neo4j://localhost:7689')







NEO4J_USER = os.getenv('NEO4J_USER', 'neo4j')







NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD', '12345678')







NEO4J_DB = os.getenv('NEO4J_DB', 'neo4j')







GPT_API_KEY = os.getenv('GPT_API_KEY')







OPENAI_MODEL = os.getenv('OPENAI_MODEL', 'gpt-5.4-nano')















BASE_DIR = Path.cwd()







PROMPTS_DIR = BASE_DIR / 'annotation_prompts'







ANNOT_DIR = BASE_DIR / 'annotation'























def get_driver():







    return GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))























def extract_question(qnum: int = 1) -> str:







    path = PROMPTS_DIR / f'q_{qnum}_prompt.txt'







    if not path.exists():







        raise FileNotFoundError(path)







    lines = path.read_text(encoding='utf-8').splitlines()







    for i, line in enumerate(lines):







        if line.strip().upper() == 'QUESTION':







            for j in range(i + 1, len(lines)):







                if lines[j].strip():







                    return lines[j].strip()







    raise RuntimeError(f'QUESTION not found in {path}')























def fetch_schema_and_scope_catalog(driver):







    with driver.session(database=NEO4J_DB) as session:







        labels = [r['l'] for r in session.run(







            'MATCH (n) UNWIND labels(n) AS l RETURN DISTINCT l AS l ORDER BY l'







        )]







        predicates = [r['p'] for r in session.run(







            'MATCH ()-[r]->() RETURN DISTINCT type(r) AS p ORDER BY p'







        )]







        cities = [r['k'] for r in session.run(







            'MATCH (c:City) WHERE c.key IS NOT NULL RETURN DISTINCT c.key AS k ORDER BY k'







        )]







        airports = [r['k'] for r in session.run(







            'MATCH (a:Airport) WHERE a.key IS NOT NULL RETURN DISTINCT a.key AS k ORDER BY k'







        )]







        countries = [r['k'] for r in session.run(







            'MATCH (c:Country) WHERE c.key IS NOT NULL RETURN DISTINCT c.key AS k ORDER BY k'







        )]







    return labels, predicates, {'cities': cities, 'airports': airports, 'countries': countries}























def normalize_text(s: str) -> str:







    return re.sub(r'[^a-z0-9]+', ' ', s.lower()).strip()























def key_aliases(key: str):







    base = key







    for pref in ('city_', 'airport_', 'country_'):







        if base.startswith(pref):







            base = base[len(pref):]







    variants = {base.lower(), base.replace('_', ' ').lower()}







    return {v for v in variants if v}























def extract_entities_from_question(question: str, scope_catalog):







    q_norm = normalize_text(question)







    out = {'cities': [], 'airports': [], 'countries': []}















    for bucket in out:







        for key in scope_catalog.get(bucket, []):







            if any(re.search(rf'\b{re.escape(alias)}\b', q_norm) for alias in key_aliases(key)):







                out[bucket].append(key)















    if re.search(r'\bparis\b', q_norm):







        for k in ['city_paris', 'city_beauvais']:







            if k in scope_catalog['cities'] and k not in out['cities']:







                out['cities'].append(k)







        for k in ['airport_cdg', 'airport_ory', 'airport_bva']:







            if k in scope_catalog['airports'] and k not in out['airports']:







                out['airports'].append(k)







        if 'country_france' in scope_catalog['countries'] and 'country_france' not in out['countries']:







            out['countries'].append('country_france')















    return {k: sorted(v) for k, v in out.items()}























def llm_generate_cypher(question: str, labels, predicates, entity_hints):







    if not GPT_API_KEY:







        raise RuntimeError('GPT_API_KEY is not set in .env')















    prompt = f"""You generate one valid Cypher query for Neo4j.







Return JSON only: {{"cypher":"<query>"}}.















Goal:







- Create ONE read-only Cypher that returns ALL relevant triples for the question.







- Output columns must be exactly: s_label, s_key, p, o_label, o_key.















Allowed schema:







- labels: {json.dumps(labels, ensure_ascii=False)}







- relationship types: {json.dumps(predicates, ensure_ascii=False)}







- entity hints: {json.dumps(entity_hints, ensure_ascii=False)}















Critical mapping rules (do not violate):







1) For relation hasDocumentCheck:







   - s must be Encounter, p='hasDocumentCheck', o must be DocumentCheck.







2) For relation requestedDocument:







   - s must be DocumentCheck, p='requestedDocument', o must be DocumentInstance.







3) documentType is a PROPERTY on DocumentInstance, not a relationship.







   - Build pseudo-triple only as:







     s_label='DocumentInstance', s_key=di.key, p='documentType', o_label='Literal', o_key=toString(di.documentType)







   - Never use (di)-[:documentType]->(...)







   - Never MATCH (:Literal)







4) For real relationships, p must equal type(r) and s/o must match the actual direction in MATCH.







   - Never remap relation to another subject/object.















Required triple families (include when present):







- Encounter -hasDocumentCheck-> DocumentCheck







- DocumentCheck -requestedDocument-> DocumentInstance







- DocumentInstance -documentType-> Literal (pseudo-triple from property)







- Encounter -atAirport-> Airport







- Encounter -atCity-> City







- Encounter -atCountry-> Country







- Airport -locatedInCity-> City







- City -locatedInCountry-> Country







- Airport -locatedInCountry-> Country















Filtering:







- Use only params: $cities, $airports, $countries.







- Empty list means no restriction.







- Use checks like: size($cities)=0 OR city.key IN $cities.







- Never use exists($param).







- Never use any($param) as map value.















Cypher quality constraints:







- No CREATE/MERGE/SET/DELETE/REMOVE/DROP.







- Query must be syntactically valid.







- Return DISTINCT rows.







- Exclude null triples.















Question:







{question}







"""















    payload = {







        'model': OPENAI_MODEL,







        'temperature': 0,







        'messages': [







            {'role': 'system', 'content': 'Return strict JSON only.'},







            {'role': 'user', 'content': prompt},







        ],







    }















    resp = requests.post(







        'https://api.openai.com/v1/chat/completions',







        headers={'Authorization': f'Bearer {GPT_API_KEY}', 'Content-Type': 'application/json'},







        data=json.dumps(payload),







        timeout=180,







    )







    resp.raise_for_status()







    content = resp.json()['choices'][0]['message']['content'].strip()















    m = re.search(r'\{.*\}', content, flags=re.S)







    if not m:







        raise RuntimeError(f'LLM response is not JSON: {content[:400]}')







    parsed = json.loads(m.group(0))







    cypher = (parsed.get('cypher') or '').strip()







    if not cypher:







        raise RuntimeError(f"LLM JSON has no non-empty 'cypher': {parsed}")















    banned = ['CREATE ', 'MERGE ', 'DELETE ', 'DETACH ', ' SET ', 'REMOVE ', 'DROP ', 'CALL dbms']







    up = f' {cypher.upper()} '







    if any(b in up for b in banned):







        raise RuntimeError('Generated Cypher is not read-only')















    invalid_param_patterns = [







        r'(?i)\bexists\s*\(\s*\$[a-z_][a-z0-9_]*\s*\)',







        r'(?i)\bany\s*\(\s*\$[a-z_][a-z0-9_]*\s*\)',







    ]







    if any(re.search(pat, cypher) for pat in invalid_param_patterns):







        raise RuntimeError('Generated Cypher uses invalid parameter functions')















    return cypher























def fallback_cypher():







    return """







MATCH (e:Encounter)







OPTIONAL MATCH (e)-[r_doc:hasDocumentCheck]->(dc:DocumentCheck)







OPTIONAL MATCH (dc)-[r_req:requestedDocument]->(di:DocumentInstance)







OPTIONAL MATCH (e)-[r_air:atAirport]->(a:Airport)







OPTIONAL MATCH (e)-[r_city:atCity]->(c:City)







OPTIONAL MATCH (e)-[r_country:atCountry]->(co:Country)







OPTIONAL MATCH (a)-[r_a_city:locatedInCity]->(ac:City)







OPTIONAL MATCH (ac)-[r_ac_co:locatedInCountry]->(aco:Country)







OPTIONAL MATCH (c)-[r_c_co:locatedInCountry]->(cco:Country)







OPTIONAL MATCH (a)-[r_a_co:locatedInCountry]->(aco2:Country)







WITH e, dc, di, a, c, co, ac, aco, cco, aco2,







     r_doc, r_req, r_air, r_city, r_country, r_a_city, r_ac_co, r_c_co, r_a_co,







     $cities AS cities, $airports AS airports, $countries AS countries







WHERE







  (size(cities) = 0 OR c.key IN cities OR ac.key IN cities)







  AND (size(airports) = 0 OR a.key IN airports)







  AND (size(countries) = 0 OR co.key IN countries OR aco.key IN countries OR cco.key IN countries OR aco2.key IN countries)







WITH [







  CASE WHEN r_doc IS NOT NULL THEN {s_label:'Encounter', s_key:coalesce(e.key, elementId(e)), p:type(r_doc), o_label:'DocumentCheck', o_key:coalesce(dc.key, elementId(dc))} END,







  CASE WHEN r_req IS NOT NULL THEN {s_label:'DocumentCheck', s_key:coalesce(dc.key, elementId(dc)), p:type(r_req), o_label:'DocumentInstance', o_key:coalesce(di.key, elementId(di))} END,







  CASE WHEN di IS NOT NULL AND di.documentType IS NOT NULL THEN {s_label:'DocumentInstance', s_key:coalesce(di.key, elementId(di)), p:'documentType', o_label:'Literal', o_key:toString(di.documentType)} END,







  CASE WHEN r_air IS NOT NULL THEN {s_label:'Encounter', s_key:coalesce(e.key, elementId(e)), p:type(r_air), o_label:'Airport', o_key:coalesce(a.key, elementId(a))} END,







  CASE WHEN r_city IS NOT NULL THEN {s_label:'Encounter', s_key:coalesce(e.key, elementId(e)), p:type(r_city), o_label:'City', o_key:coalesce(c.key, elementId(c))} END,







  CASE WHEN r_country IS NOT NULL THEN {s_label:'Encounter', s_key:coalesce(e.key, elementId(e)), p:type(r_country), o_label:'Country', o_key:coalesce(co.key, elementId(co))} END,







  CASE WHEN r_a_city IS NOT NULL THEN {s_label:'Airport', s_key:coalesce(a.key, elementId(a)), p:type(r_a_city), o_label:'City', o_key:coalesce(ac.key, elementId(ac))} END,







  CASE WHEN r_ac_co IS NOT NULL THEN {s_label:'City', s_key:coalesce(ac.key, elementId(ac)), p:type(r_ac_co), o_label:'Country', o_key:coalesce(aco.key, elementId(aco))} END,







  CASE WHEN r_c_co IS NOT NULL THEN {s_label:'City', s_key:coalesce(c.key, elementId(c)), p:type(r_c_co), o_label:'Country', o_key:coalesce(cco.key, elementId(cco))} END,







  CASE WHEN r_a_co IS NOT NULL THEN {s_label:'Airport', s_key:coalesce(a.key, elementId(a)), p:type(r_a_co), o_label:'Country', o_key:coalesce(aco2.key, elementId(aco2))} END







] AS triple_candidates







UNWIND triple_candidates AS t







WITH t WHERE t IS NOT NULL







RETURN DISTINCT t.s_label AS s_label, t.s_key AS s_key, t.p AS p, t.o_label AS o_label, t.o_key AS o_key







"""























def run_retrieval(driver, cypher: str, params: dict):







    with driver.session(database=NEO4J_DB) as session:







        rows = session.run(cypher, **params).data()







    out = []







    seen = set()







    for r in rows:







        t = {







            's_label': r.get('s_label'),







            's_key': r.get('s_key'),







            'p': r.get('p'),







            'o_label': r.get('o_label'),







            'o_key': r.get('o_key'),







        }







        key = (t['s_label'], t['s_key'], t['p'], t['o_label'], t['o_key'])







        if key in seen:







            continue







        seen.add(key)







        out.append(t)







    return out























def validate_cypher(driver, cypher: str, params: dict):







    with driver.session(database=NEO4J_DB) as session:







        session.run('EXPLAIN ' + cypher, **params).consume()























def normalize_triple(t):







    if 's_label' in t:







        return (t['s_label'], t['s_key'], t['p'], t['o_label'], t['o_key'])







    s = t.get('s', {})







    o = t.get('o', {})







    return (s.get('label'), s.get('key'), t.get('p'), o.get('label'), o.get('key'))























def triples_to_set(triples):







    return {normalize_triple(t) for t in triples}























def load_gold(qnum: int = 1):







    triples = []







    base = ANNOT_DIR / f'q_{qnum}'







    for folder in ['fr_1', 'fr_2', 'germany', 'italy', 'spain']:







        p = base / folder







        if not p.exists():







            continue







        for f in sorted(p.glob('annotation_*.json')):







            data = json.loads(f.read_text(encoding='utf-8'))







            triples.extend(data.get('triples', []))







    return triples























question = extract_question(1)















with get_driver() as driver:







    labels, predicates, scope_catalog = fetch_schema_and_scope_catalog(driver)















entity_hints = extract_entities_from_question(question, scope_catalog)







params = {







    'cities': entity_hints['cities'],







    'airports': entity_hints['airports'],







    'countries': entity_hints['countries'],







}















print('Q1:', question)







print('Labels:', labels)







print('Predicates:', predicates)







print('Entity hints:', entity_hints)















try:







    generated_cypher = llm_generate_cypher(question, labels, predicates, entity_hints)







    cypher_source = 'LLM'







except Exception as e:







    print('LLM generation failed, fallback will be used:', e)







    generated_cypher = fallback_cypher()







    cypher_source = 'fallback'















with get_driver() as driver:







    try:







        validate_cypher(driver, generated_cypher, params)







        retrieved = run_retrieval(driver, generated_cypher, params)







    except Exception as e:







        print('Generated Cypher failed validation/execution, fallback will be used:', e)







        generated_cypher = fallback_cypher()







        cypher_source = 'fallback_after_validation'







        validate_cypher(driver, generated_cypher, params)







        retrieved = run_retrieval(driver, generated_cypher, params)















print('\nCypher source:', cypher_source)







print('Generated Cypher:\n', generated_cypher)















gold = load_gold(1)







retrieved_set = triples_to_set(retrieved)







gold_set = triples_to_set(gold)















tp = len(retrieved_set & gold_set)







precision = tp / len(retrieved_set) if retrieved_set else 0.0







recall = tp / len(gold_set) if gold_set else 0.0







f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0















print('\nMetrics vs gold (Q1):')







print(f'retrieved={len(retrieved_set)} gold={len(gold_set)} tp={tp}')







print(f'precision={precision:.6f} recall={recall:.6f} f1={f1:.6f}')















print('\nSample retrieved triples (first 20):')







for t in retrieved[:20]:







    print(t)


In [19]:
import json
import os
from pathlib import Path

from neo4j import GraphDatabase

NEO4J_URI = os.getenv('NEO4J_URI', 'neo4j://localhost:7689')
NEO4J_USER = os.getenv('NEO4J_USER', 'neo4j')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD', '12345678')
NEO4J_DB = os.getenv('NEO4J_DB', 'neo4j')

BASE_DIR = Path.cwd()
ANNOT_DIR = BASE_DIR / 'annotation'
STORE_PATH = BASE_DIR / 'ideal_cypher_queries.json'

QNUM = 1  # change to select question number

if not STORE_PATH.exists():
    raise FileNotFoundError(STORE_PATH)

store = json.loads(STORE_PATH.read_text(encoding='utf-8'))
entry = store.get(str(QNUM)) or store.get(QNUM)
if not entry:
    raise KeyError(f'No entry for question {QNUM} in {STORE_PATH}')

cypher = (entry.get('cypher') or '').strip()
if not cypher:
    raise ValueError(f'Empty cypher for question {QNUM}')


def get_driver():
    return GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))


def validate_cypher(driver, cypher_text):
    with driver.session(database=NEO4J_DB) as session:
        session.run('EXPLAIN ' + cypher_text).consume()


def run_retrieval(driver, cypher_text):
    with driver.session(database=NEO4J_DB) as session:
        rows = session.run(cypher_text).data()
    out = []
    seen = set()
    for r in rows:
        t = {
            's_label': r.get('s_label'),
            's_key': r.get('s_key'),
            'p': r.get('p'),
            'o_label': r.get('o_label'),
            'o_key': r.get('o_key'),
        }
        key = (t['s_label'], t['s_key'], t['p'], t['o_label'], t['o_key'])
        if key in seen:
            continue
        seen.add(key)
        out.append(t)
    return out


def normalize_triple(t):
    if 's_label' in t:
        return (t['s_label'], t['s_key'], t['p'], t['o_label'], t['o_key'])
    s = t.get('s', {})
    o = t.get('o', {})
    return (s.get('label'), s.get('key'), t.get('p'), o.get('label'), o.get('key'))


def triples_to_set(triples):
    return {normalize_triple(t) for t in triples}


def load_gold(qnum: int = 1):
    triples = []
    base = ANNOT_DIR / f'q_{qnum}'
    for folder in ['fr_1', 'fr_2', 'germany', 'italy', 'spain']:
        p = base / folder
        if not p.exists():
            continue
        for f in sorted(p.glob('annotation_*.json')):
            data = json.loads(f.read_text(encoding='utf-8'))
            triples.extend(data.get('triples', []))
    return triples


with get_driver() as driver:
    validate_cypher(driver, cypher)
    retrieved = run_retrieval(driver, cypher)

gold = load_gold(QNUM)
retrieved_set = triples_to_set(retrieved)
gold_set = triples_to_set(gold)

tp = len(retrieved_set & gold_set)
precision = tp / len(retrieved_set) if retrieved_set else 0.0
recall = tp / len(gold_set) if gold_set else 0.0
f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0

print('\nQuestion:', entry.get('question'))
print('Metrics vs gold (Q{QNUM}):'.format(QNUM=QNUM))
print(f'retrieved={len(retrieved_set)} gold={len(gold_set)} tp={tp}')
print(f'precision={precision:.6f} recall={recall:.6f} f1={f1:.6f}')

# print('\nSample retrieved triples (first 20):')
# for t in retrieved[:20]:
#     print(t)



Question: Which documents do I need to bring to passport control in Paris?
Metrics vs gold (Q1):
retrieved=678 gold=645 tp=645
precision=0.951327 recall=1.000000 f1=0.975057


In [21]:
print(gold_set)

{('Encounter', 'encounter_11888385_author_2', 'hasDocumentCheck', 'DocumentCheck', 'doccheck_encounter_11888385_author_2_accommodation_proof'), ('Encounter', 'encounter_12214016_woman_1', 'atAirport', 'Airport', 'airport_ory'), ('Encounter', 'encounter_11888385_author_1', 'atCity', 'City', 'city_paris'), ('DocumentCheck', 'doccheck_encounter_9684707_author_1_foreign_passport', 'requestedDocument', 'DocumentInstance', 'documentinstance_encounter_9684707_author_1_foreign_passport'), ('DocumentInstance', 'documentinstance_encounter_12134610_husband_2_foreign_passport', 'documentType', 'Literal', 'foreign_passport'), ('Encounter', 'encounter_11883695_author_1', 'hasDocumentCheck', 'DocumentCheck', 'doccheck_encounter_11883695_author_1_foreign_passport'), ('DocumentCheck', 'doccheck_encounter_11927193_woman_1_foreign_passport', 'requestedDocument', 'DocumentInstance', 'documentinstance_encounter_11927193_woman_1_foreign_passport'), ('Encounter', 'encounter_12117020_friend_1', 'atAirport', '

#  Цепочки и интенты

In [84]:
import json
import os
from pathlib import Path

import requests
from neo4j import GraphDatabase

NEO4J_URI = os.getenv('NEO4J_URI', 'neo4j://localhost:7689')
NEO4J_USER = os.getenv('NEO4J_USER', 'neo4j')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD', '12345678')
NEO4J_DB = os.getenv('NEO4J_DB', 'neo4j')

GPT_API_KEY = os.getenv('GPT_API_KEY')
GPT_MODEL = os.getenv('GPT_MODEL', 'gpt-5.4-nano')

BASE_DIR = Path.cwd()
ANNOT_DIR = BASE_DIR / 'annotation'

INTENTS = {
    'documentCheck': {
        'chains': [
            'Encounter-[:hasDocumentCheck]->DocumentCheck-[:requestedDocument]->DocumentInstance-[:documentType]->Literal',
            'Encounter-[:atAirport]->Airport-[:locatedInCity]->City-[:locatedInCountry]->Country',
            'Encounter-[:atCity]->City-[:locatedInCountry]->Country',
            'Encounter-[:atCountry]->Country'
        ]
    },
    'questioning': {
        'chains': [
            'Encounter-[:hasQuestioning]->Questioning-[:topicKey]->Literal',
            'Encounter-[:atAirport]->Airport-[:locatedInCity]->City-[:locatedInCountry]->Country',
            'Encounter-[:atCity]->City-[:locatedInCountry]->Country',
            'Encounter-[:atCountry]->Country'
        ]
    }
}

QUESTION_NUMS = [1, 2, 62, 87]
INTENT_BY_QNUM = {
    1: ['documentCheck'],
    2: ['documentCheck'],
    62: ['questioning'],
    87: ['documentCheck', 'questioning']
}



def get_question_text(qnum: int) -> str:
    base = ANNOT_DIR / f'q_{qnum}'
    for folder in ['fr_1', 'fr_2', 'germany', 'italy', 'spain']:
        p = base / folder
        if not p.exists():
            continue
        for f in sorted(p.glob('annotation_*.json')):
            data = json.loads(f.read_text(encoding='utf-8'))
            q = (data.get('question') or '').strip()
            if q:
                return q
    raise FileNotFoundError(f'Question text not found for q_{qnum}')


def build_documentcheck_cypher():
    return '''
MATCH (e:Encounter)
OPTIONAL MATCH (e)-[:atAirport]->(a:Airport)
OPTIONAL MATCH (e)-[:atCity]->(c:City)
OPTIONAL MATCH (e)-[:atCountry]->(co:Country)
OPTIONAL MATCH (a)-[:locatedInCity]->(c2:City)
OPTIONAL MATCH (a)-[:locatedInCountry]->(co2:Country)
OPTIONAL MATCH (c)-[:locatedInCountry]->(co3:Country)
WITH e,a,c,co,c2,co2,co3,
     (size($airports) > 0) AS has_airports,
     (size($cities) > 0) AS has_cities,
     (size($countries) > 0) AS has_countries,
     (a IS NOT NULL AND a.key IN $airports) AS match_airport,
     ((c IS NOT NULL AND c.key IN $cities) OR (c2 IS NOT NULL AND c2.key IN $cities)) AS match_city,
     ((co IS NOT NULL AND co.key IN $countries) OR (co2 IS NOT NULL AND co2.key IN $countries) OR (co3 IS NOT NULL AND co3.key IN $countries)) AS match_country
WHERE CASE
    WHEN (has_airports OR has_cities) THEN (match_airport OR match_city)
    WHEN has_countries THEN match_country
    ELSE true
END
OPTIONAL MATCH (e)-[:hasDocumentCheck]->(dc:DocumentCheck)
OPTIONAL MATCH (dc)-[:requestedDocument]->(di:DocumentInstance)
WITH e,a,c,co,c2,co2,co3,dc,di,di.documentType AS di_type
CALL (e,dc,di,di_type,a,c,co,c2,co2,co3) {
  WITH e,dc
  WHERE e IS NOT NULL AND dc IS NOT NULL
  RETURN DISTINCT labels(e)[0] AS s_label, e.key AS s_key, 'hasDocumentCheck' AS p, labels(dc)[0] AS o_label, dc.key AS o_key
  UNION
  WITH dc,di
  WHERE dc IS NOT NULL AND di IS NOT NULL
  RETURN DISTINCT labels(dc)[0] AS s_label, dc.key AS s_key, 'requestedDocument' AS p, labels(di)[0] AS o_label, di.key AS o_key
  UNION
  WITH di,di_type
  WHERE di IS NOT NULL AND di_type IS NOT NULL
  RETURN DISTINCT labels(di)[0] AS s_label, di.key AS s_key, 'documentType' AS p, 'Literal' AS o_label, di_type AS o_key
  UNION
  WITH e,a
  WHERE e IS NOT NULL AND a IS NOT NULL
  RETURN DISTINCT labels(e)[0] AS s_label, e.key AS s_key, 'atAirport' AS p, labels(a)[0] AS o_label, a.key AS o_key
  UNION
  WITH e,c
  WHERE e IS NOT NULL AND c IS NOT NULL
  RETURN DISTINCT labels(e)[0] AS s_label, e.key AS s_key, 'atCity' AS p, labels(c)[0] AS o_label, c.key AS o_key
  UNION
  WITH e,co
  WHERE e IS NOT NULL AND co IS NOT NULL
  RETURN DISTINCT labels(e)[0] AS s_label, e.key AS s_key, 'atCountry' AS p, labels(co)[0] AS o_label, co.key AS o_key
  UNION
  WITH a,c2
  WHERE a IS NOT NULL AND c2 IS NOT NULL
  RETURN DISTINCT labels(a)[0] AS s_label, a.key AS s_key, 'locatedInCity' AS p, labels(c2)[0] AS o_label, c2.key AS o_key
  UNION
  WITH c,co3
  WHERE c IS NOT NULL AND co3 IS NOT NULL
  RETURN DISTINCT labels(c)[0] AS s_label, c.key AS s_key, 'locatedInCountry' AS p, labels(co3)[0] AS o_label, co3.key AS o_key
  UNION
  WITH a,co2
  WHERE a IS NOT NULL AND co2 IS NOT NULL
  RETURN DISTINCT labels(a)[0] AS s_label, a.key AS s_key, 'locatedInCountry' AS p, labels(co2)[0] AS o_label, co2.key AS o_key
}
WITH s_label, s_key, p, o_label, o_key
WHERE s_label IS NOT NULL AND s_key IS NOT NULL AND o_label IS NOT NULL AND o_key IS NOT NULL
RETURN DISTINCT s_label, s_key, p, o_label, o_key
'''.strip()


def build_questioning_cypher():
    return '''
MATCH (e:Encounter)
OPTIONAL MATCH (e)-[:atAirport]->(a:Airport)
OPTIONAL MATCH (e)-[:atCity]->(c:City)
OPTIONAL MATCH (e)-[:atCountry]->(co:Country)
OPTIONAL MATCH (a)-[:locatedInCity]->(c2:City)
OPTIONAL MATCH (a)-[:locatedInCountry]->(co2:Country)
OPTIONAL MATCH (c)-[:locatedInCountry]->(co3:Country)
WITH e,a,c,co,c2,co2,co3,
     (size($airports) > 0) AS has_airports,
     (size($cities) > 0) AS has_cities,
     (size($countries) > 0) AS has_countries,
     (a IS NOT NULL AND a.key IN $airports) AS match_airport,
     ((c IS NOT NULL AND c.key IN $cities) OR (c2 IS NOT NULL AND c2.key IN $cities)) AS match_city,
     ((co IS NOT NULL AND co.key IN $countries) OR (co2 IS NOT NULL AND co2.key IN $countries) OR (co3 IS NOT NULL AND co3.key IN $countries)) AS match_country
WHERE CASE
    WHEN (has_airports OR has_cities) THEN (match_airport OR match_city)
    WHEN has_countries THEN match_country
    ELSE true
END
MATCH (e)-[:hasQuestioning]->(q:Questioning)
WITH e,a,c,co,c2,co2,co3,q,q.topicKey AS q_topic
CALL (e,q,q_topic,a,c,co,c2,co2,co3) {
  WITH e,q
  WHERE e IS NOT NULL AND q IS NOT NULL
  RETURN DISTINCT labels(e)[0] AS s_label, e.key AS s_key, 'hasQuestioning' AS p, labels(q)[0] AS o_label, q.key AS o_key
  UNION
  WITH q,q_topic
  WHERE q IS NOT NULL AND q_topic IS NOT NULL
  RETURN DISTINCT labels(q)[0] AS s_label, q.key AS s_key, 'topicKey' AS p, 'Literal' AS o_label, q_topic AS o_key
  UNION
  WITH e,a
  WHERE e IS NOT NULL AND a IS NOT NULL
  RETURN DISTINCT labels(e)[0] AS s_label, e.key AS s_key, 'atAirport' AS p, labels(a)[0] AS o_label, a.key AS o_key
  UNION
  WITH e,c
  WHERE e IS NOT NULL AND c IS NOT NULL
  RETURN DISTINCT labels(e)[0] AS s_label, e.key AS s_key, 'atCity' AS p, labels(c)[0] AS o_label, c.key AS o_key
  UNION
  WITH e,co
  WHERE e IS NOT NULL AND co IS NOT NULL
  RETURN DISTINCT labels(e)[0] AS s_label, e.key AS s_key, 'atCountry' AS p, labels(co)[0] AS o_label, co.key AS o_key
  UNION
  WITH a,c2
  WHERE a IS NOT NULL AND c2 IS NOT NULL
  RETURN DISTINCT labels(a)[0] AS s_label, a.key AS s_key, 'locatedInCity' AS p, labels(c2)[0] AS o_label, c2.key AS o_key
  UNION
  WITH c,co3
  WHERE c IS NOT NULL AND co3 IS NOT NULL
  RETURN DISTINCT labels(c)[0] AS s_label, c.key AS s_key, 'locatedInCountry' AS p, labels(co3)[0] AS o_label, co3.key AS o_key
  UNION
  WITH a,co2
  WHERE a IS NOT NULL AND co2 IS NOT NULL
  RETURN DISTINCT labels(a)[0] AS s_label, a.key AS s_key, 'locatedInCountry' AS p, labels(co2)[0] AS o_label, co2.key AS o_key
}
WITH s_label, s_key, p, o_label, o_key
WHERE s_label IS NOT NULL AND s_key IS NOT NULL AND o_label IS NOT NULL AND o_key IS NOT NULL
RETURN DISTINCT s_label, s_key, p, o_label, o_key
'''.strip()

def get_driver():
    return GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))


def run_retrieval(driver, cypher_text, params):
    with driver.session(database=NEO4J_DB) as session:
        rows = session.run(cypher_text, params).data()
    out = []
    seen = set()
    for r in rows:
        t = (r.get('s_label'), r.get('s_key'), r.get('p'), r.get('o_label'), r.get('o_key'))
        if t in seen:
            continue
        seen.add(t)
        out.append(t)
    return out


def load_gold(qnum: int):
    triples = []
    base = ANNOT_DIR / f'q_{qnum}'
    for folder in ['fr_1', 'fr_2', 'germany', 'italy', 'spain']:
        p = base / folder
        if not p.exists():
            continue
        for f in sorted(p.glob('annotation_*.json')):
            data = json.loads(f.read_text(encoding='utf-8'))
            for t in data.get('triples', []):
                s = t.get('s', {})
                o = t.get('o', {})
                triples.append((s.get('label'), s.get('key'), t.get('p'), o.get('label'), o.get('key')))
    return set(triples)


def fetch_candidates(driver):
    def fetch(label):
        q = f"MATCH (n:{label}) RETURN n.key AS key"
        with driver.session(database=NEO4J_DB) as session:
            return [r['key'] for r in session.run(q).data() if r.get('key')]

    return {
        'cities': sorted(fetch('City')),
        'airports': sorted(fetch('Airport')),
        'countries': sorted(fetch('Country'))
    }


def llm_select_filters(question: str, candidates: dict) -> dict:
    if not GPT_API_KEY:
        raise RuntimeError('GPT_API_KEY is not set')

    system = (
        'You select graph filter keys for a question. '
        'Return ONLY a JSON object with keys: cities, airports, countries. '
        'Values are arrays of keys taken strictly from the provided candidates. '
        'If none apply, use empty arrays. ''If the question is about a country and does not mention a specific city or airport, use only countries and leave cities/airports empty. ''BVA (Beauvais) is a Paris-area airport; for Paris questions, include airport_bva.'
    )

    user = {
        'question': question,
        'candidates': candidates
    }

    schema = {
        'type': 'object',
        'additionalProperties': False,
        'required': ['cities', 'airports', 'countries'],
        'properties': {
            'cities': {'type': 'array', 'items': {'type': 'string'}},
            'airports': {'type': 'array', 'items': {'type': 'string'}},
            'countries': {'type': 'array', 'items': {'type': 'string'}}
        }
    }

    payload = {
        'model': GPT_MODEL,
        'input': [
            {'role': 'system', 'content': system},
            {'role': 'user', 'content': json.dumps(user, ensure_ascii=False)}
        ],
        'text': {
            'format': {
                'type': 'json_schema',
                'name': 'filters',
                'schema': schema,
                'strict': True
            }
        }
    }

    resp = requests.post(
        'https://api.openai.com/v1/responses',
        headers={'Authorization': f'Bearer {GPT_API_KEY}'},
        json=payload,
        timeout=60
    )
    if not resp.ok:
        raise RuntimeError(f'OpenAI API error {resp.status_code}: {resp.text[:500]}')
    data = resp.json()
    text = data.get('output_text') or ''
    if not text:
        out = []
        for item in data.get('output', []):
            for content in item.get('content', []):
                if content.get('type') == 'output_text':
                    out.append(content.get('text', ''))
        text = ''.join(out).strip()
    if not text:
        raise ValueError('Empty LLM response')
    try:
        result = json.loads(text)
    except json.JSONDecodeError as e:
        raise ValueError(f'LLM did not return valid JSON: {text[:200]}') from e
    return result


def sanitize_filters(filters: dict, candidates: dict) -> dict:
    out = {'cities': [], 'airports': [], 'countries': []}
    for k in out:
        vals = filters.get(k, []) if isinstance(filters, dict) else []
        if not isinstance(vals, list):
            vals = []
        allowed = set(candidates.get(k, []))
        out[k] = sorted({v for v in vals if v in allowed})
    return out


cypher = build_documentcheck_cypher()

print('Intents:', ', '.join(INTENTS.keys()))
for name, spec in INTENTS.items():
    print(f'{name} chains:')
    for ch in spec['chains']:
        print('-', ch)

with get_driver() as driver:
    candidates = fetch_candidates(driver)
    for qnum in QUESTION_NUMS:
        question = get_question_text(qnum)
        intents = INTENT_BY_QNUM.get(qnum, [])
        raw_filters = llm_select_filters(question, candidates)
        filters = sanitize_filters(raw_filters, candidates)
        if not intents:
            raise ValueError(f'Intent not set for question {qnum}')
        retrieved = set()
        for intent in intents:
            if intent == 'documentCheck':
                cypher = build_documentcheck_cypher()
            elif intent == 'questioning':
                cypher = build_questioning_cypher()
            else:
                raise ValueError(f'Unsupported intent: {intent}')
            triples = run_retrieval(driver, cypher, filters)
            retrieved.update(triples)
        gold = load_gold(qnum)
        # retrieved already accumulated
        tp = len(retrieved & gold)
        precision = tp / len(retrieved) if retrieved else 0.0
        recall = tp / len(gold) if gold else 0.0
        f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0
        print(f"Q{qnum}: {question}")
        print(f"filters={filters}")
        print(f"retrieved={len(retrieved)} gold={len(gold)} tp={tp}")
        print(f"precision={precision:.6f} recall={recall:.6f} f1={f1:.6f}")


Intents: documentCheck, questioning
documentCheck chains:
- Encounter-[:hasDocumentCheck]->DocumentCheck-[:requestedDocument]->DocumentInstance-[:documentType]->Literal
- Encounter-[:atAirport]->Airport-[:locatedInCity]->City-[:locatedInCountry]->Country
- Encounter-[:atCity]->City-[:locatedInCountry]->Country
- Encounter-[:atCountry]->Country
questioning chains:
- Encounter-[:hasQuestioning]->Questioning-[:topicKey]->Literal
- Encounter-[:atAirport]->Airport-[:locatedInCity]->City-[:locatedInCountry]->Country
- Encounter-[:atCity]->City-[:locatedInCountry]->Country
- Encounter-[:atCountry]->Country
Q1: Which documents do I need to bring to passport control in Paris?
filters={'cities': ['city_paris'], 'airports': ['airport_bva', 'airport_cdg', 'airport_ory'], 'countries': ['country_france']}
retrieved=678 gold=678 tp=678
precision=1.000000 recall=1.000000 f1=1.000000
Q2: Which documents do I need to bring to passport control in Nice?
filters={'cities': ['city_nice'], 'airports': ['airp